# 📚 Data Preprocessing with Hugging Face Datasets
**Tutorial Reference:** [HuggingFace LLM Course — Chapter 5.3: Time to Slice and Dice](https://huggingface.co/learn/llm-course/chapter5/3)

---

## 🗂️ Overview

In the real world, data is messy. Before you can train a model, you almost always need to **clean, filter, and transform** your dataset. The `🤗 Datasets` library gives you a powerful, Pandas-like API to do this efficiently — even on datasets too large to fit in memory.

In this notebook, we work with the [Drug Review Dataset](https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+(Drugs.com)) from the UCI ML Repository. It contains:
- **Patient reviews** of drugs
- **Condition** being treated
- **Rating** (1–10 scale)

We will progressively clean and preprocess this dataset through the following steps:

| Step | Task |
|------|------|
| 1 | Download & load the dataset |
| 2 | Inspect a random sample |
| 3 | Verify and rename columns |
| 4 | Normalize text (lowercase) |
| 5 | Compute & filter by review length |
| 6 | Unescape HTML characters |
| 7 | Tokenize with fast & slow tokenizers |
| 8 | Multiprocessing with `num_proc` |

---

## 🔧 Step 0: Install Required Libraries

We need 3 libraries:
- **`datasets`** — the core HuggingFace Datasets library for loading and manipulating datasets
- **`evaluate`** — for computing metrics (used later)
- **`transformers[sentencepiece]`** — for tokenizers and models; `sentencepiece` is needed for some tokenizers

> **Note:** Run this cell only once. If already installed, you can skip it.


In [2]:
!pip install datasets evaluate transformers[sentencepiece]


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

## 📥 Step 1: Download the Drug Review Dataset

The dataset is hosted as a `.zip` file at the UCI ML Repository. We use Python's `requests` and `zipfile` libraries to:

1. **Download** the zip file into memory (no saving to disk first)
2. **Extract** the contents into a `drugs_data/` folder

The dataset contains two TSV (Tab-Separated Values) files:
- `drugsComTrain_raw.tsv` — Training data (~161,000 rows)
- `drugsComTest_raw.tsv` — Test data (~53,000 rows)

> **Why TSV?** TSV is a common data format. It's identical to CSV but uses a **tab character (`\t`)** instead of a comma as the delimiter. This avoids issues when the data itself contains commas (e.g., in drug names or reviews).


In [3]:
import requests
import zipfile
from io import BytesIO

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"

response = requests.get(url)
zip_file = zipfile.ZipFile(BytesIO(response.content))
zip_file.extractall("drugs_data")

print("Download and extraction complete")

Download and extraction complete


---

## 📂 Step 2: Load the Dataset with `load_dataset()`

`🤗 Datasets` can load CSV/TSV files directly using its built-in `csv` loader script.

```python
load_dataset("csv", data_files=data_files, delimiter="\t")
```

**Key points:**
- We pass a **dictionary** `{"train": ..., "test": ...}` to `data_files` so the library knows which file is which split
- `delimiter="\t"` tells the parser to use tab as the separator (since these are TSV files)
- The result is a `DatasetDict` — a dictionary-like object containing multiple `Dataset` splits

**What is a `DatasetDict`?**
```
DatasetDict({
    train: Dataset({...}),
    test:  Dataset({...})
})
```
Think of it like a Python dict where each value is a full dataset split. You access splits with `drug_dataset["train"]` or `drug_dataset["test"]`.

> **`SyntaxWarning` about `\d`:** You may see a warning about an invalid escape sequence in the file path. This happens on Windows when backslashes in strings are not escaped properly. It doesn't break anything, but using raw strings (`r"..."`) or forward slashes is best practice.


In [4]:
from datasets import load_dataset


data_files = {"train": "drugs_data\drugsComTrain_raw.tsv", "test": "drugs_data\drugsComTest_raw.tsv"}
# \t is the tab character in Python
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
C:\Users\sujat\AppData\Local\Temp\ipykernel_25180\161520518.py:4: SyntaxWarning: invalid escape sequence '\d'
  data_files = {"train": "drugs_data\drugsComTrain_raw.tsv", "test": "drugs_data\drugsComTest_raw.tsv"}
C:\Users\sujat\AppData\Local\Temp\ipykernel_25180\161520518.py:4: SyntaxWarning: invalid escape sequence '\d'
  data_files = {"train": "drugs_data\drugsComTrain_raw.tsv", "test": "drugs_data\drugsComTest_raw.tsv"}
c:\Users\sujat\projects\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 161297 examples [00:02, 65150.01 examples/s]
Generating test split: 53766 examples [00:00, 6

---

## 🔍 Step 3: Inspect a Random Sample

Before doing anything with a dataset, **always look at a small sample first!** This helps you understand the structure and catch potential issues early.

### How `shuffle` + `select` works:

```python
drug_dataset["train"].shuffle(seed=42).select(range(1000))
```

- **`.shuffle(seed=42)`** — Randomly reorders all rows. Using a fixed `seed` ensures you get the **same "random" order every time** (reproducibility). Without a seed, you'd get different rows each run.
- **`.select(range(1000))`** — Picks rows at indices 0 through 999 from the shuffled dataset (i.e., a random 1000-row subset)
- **`[:3]`** — Then slices the first 3 rows from that sample to display

### What to look for in the output:
1. **`Unnamed: 0`** — Looks like a patient ID (anonymized integer). Should be renamed.
2. **`condition`** — Mixed casing (`"Gout, Acute"`, `"ibromyalgia"`) — needs normalization.
3. **`review`** — Contains HTML codes like `&#039;` (which means `'`) and Windows line endings `\r\n` — needs cleaning.
4. **`rating`** — Floats from 1.0 to 10.0 — looks clean.


In [5]:
drug_sample = drug_dataset["train"].shuffle(seed=42).select(range(1000))
# Print the first few examples of the dataset
drug_sample[:3]

{'Unnamed: 0': [87571, 178045, 80482],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic'],
 'condition': ['Gout, Acute', 'ibromyalgia', 'Inflammatory Conditions'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',
  '"I have been taking Mobic for over a year with no side effects other than 

---

## ✅ Step 4: Verify Patient IDs are Unique

We suspect `Unnamed: 0` is a unique patient identifier. Let's **verify this hypothesis** using `Dataset.unique()`.

```python
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))
```

**How this works:**
- `drug_dataset.keys()` → `["train", "test"]`
- `drug_dataset[split].unique("Unnamed: 0")` → returns a list of all **unique values** in that column
- If `len(unique values) == len(total rows)`, then every row has a **different ID** → it's a unique patient identifier ✅

**Why does this matter?**
- Confirms no duplicate patients are in the dataset
- Justifies renaming the column to something meaningful like `patient_id`
- If the assertion failed (duplicates existed), we might need to aggregate or deduplicate

> **`assert` in Python:** Think of it as a "sanity check". If the condition is `False`, it raises an `AssertionError` and stops execution. It's a lightweight way to validate assumptions about your data.


In [6]:
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))

---

## ✏️ Step 5: Rename Column — `Unnamed: 0` → `patient_id`

Now that we've confirmed `Unnamed: 0` is a unique patient ID, let's rename it to something meaningful.

```python
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)
```

**Key points:**
- `rename_column()` is called on the **`DatasetDict`** — it renames the column in **both** `train` and `test` splits simultaneously
- This is **non-destructive** — it returns a new `DatasetDict` (we reassign to `drug_dataset`)
- The column order is preserved; only the name changes

**Why rename at all?**
- `Unnamed: 0` is an auto-generated name from pandas/CSV export — it carries no semantic meaning
- Using descriptive column names makes code more readable and reduces bugs

After renaming, the features list becomes:  
`['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount']`


In [7]:
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

---

## 🔡 Step 6: Normalize `condition` Column to Lowercase

The `condition` column has inconsistent casing (e.g., `"Birth Control"`, `"birth control"`, `"BIRTH CONTROL"` could all mean the same thing). We normalize it to lowercase.

### The `Dataset.map()` method

`map()` is the **Swiss Army knife** of dataset processing. It applies a function to **every row** in the dataset.

```python
def lowercase_condition(example):
    return {"condition": example["condition"].lower() if example["condition"] is not None else None}
```

**How it works:**
- `example` is a **dictionary** representing one row: `{"patient_id": ..., "drugName": ..., "condition": ..., ...}`
- We return a **dictionary of only the columns we want to update** — `map()` will merge this back into the full row
- The `if ... else None` guard handles `None` values (some rows have no condition listed)

**What happens under the hood:**
1. HuggingFace iterates through every row
2. Calls your function on each row dict
3. Updates the dataset with your returned values
4. Returns a new `Dataset` (original is unchanged unless you reassign)

> **Note on `None` values:** The original tutorial code `example["condition"].lower()` would crash on `None` entries with `AttributeError: 'NoneType' object has no attribute 'lower'`. We added the `None` guard here, and will properly filter `None` rows in the next step.


In [9]:
def lowercase_condition(example):
    return {"condition": example["condition"].lower() if example["condition"] is not None else None}


drug_dataset.map(lowercase_condition)

Map:   0%|          | 0/53766 [00:00<?, ? examples/s]

Map: 100%|██████████| 53766/53766 [00:06<00:00, 8499.65 examples/s] 


DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

---

## 🐍 Step 7a: Lambda Functions — A Quick Primer

Before we filter `None` values, let's understand **lambda functions** — they're heavily used with `map()` and `filter()`.

### What is a lambda function?

A lambda is an **anonymous (nameless), single-expression function**. Syntax:

```python
lambda <arguments> : <expression>
```

Equivalent to:
```python
def <anonymous>(arguments):
    return expression
```

### Examples:

```python
# Square a number
(lambda x: x * x)(3)       # → 9

# Area of a triangle  
(lambda base, height: 0.5 * base * height)(4, 8)  # → 16.0

# Filter: keep only non-None conditions
lambda x: x["condition"] is not None
```

### When to use lambdas:
- ✅ Short, one-off operations that don't need a name
- ✅ Inline with `map()` and `filter()` calls
- ❌ Complex logic (use a named `def` function instead — more readable)


In [ ]:
def filter_nones(x):
    return x["condition"] is not None

In [ ]:
(lambda x: x * x*2)(3)

18

In [ ]:
(lambda base, height: 0.5 * base * height)(4, 8)

16.0

---

## 🚫 Step 7b: Filter Out `None` Condition Rows

Some reviews have no condition listed (`None`). We need to remove them before lowercasing — you can't call `.lower()` on `None`.

```python
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)
```

**How `Dataset.filter()` works:**
- Like `map()`, but your function must return **`True` or `False`**
- Rows where the function returns `True` are **kept**
- Rows where the function returns `False` are **dropped**

**The lambda breakdown:**
```python
lambda x: x["condition"] is not None
#   x = one row (dict)
#   x["condition"] = the condition value for that row
#   is not None  = True if the condition exists, False if it's None
```

**Impact:** This removes a small number of rows (~899 train, ~295 test) with missing conditions. After this, we can safely lowercase all remaining values.

> **`filter()` vs `map()`:**
> | Method | Input | Output | Use When |
> |--------|-------|--------|----------|
> | `map()` | row dict | updated row dict | Transforming/adding columns |
> | `filter()` | row dict | True/False | Removing unwanted rows |


In [ ]:
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)

Filter: 100%|██████████| 53766/53766 [00:00<00:00, 61152.01 examples/s]


---

## ✅ Step 7c: Apply Lowercasing (Now Safe After Filtering)

With `None` rows removed, we can now safely apply `lowercase_condition` without any errors.

```python
drug_dataset = drug_dataset.map(lowercase_condition)
drug_dataset["train"]["condition"][:3]
```

**Result:** `['left ventricular dysfunction', 'adhd', 'birth control']`

All conditions are now consistently lowercase. This is important because:
- `"Birth Control"` and `"birth control"` are the same condition — without normalization, they'd be treated as different labels
- Consistent casing reduces vocabulary size during tokenization
- Makes downstream filtering, grouping, and analysis easier


In [ ]:
drug_dataset = drug_dataset.map(lowercase_condition)
# Check that lowercasing worked
drug_dataset["train"]["condition"][:3]

Map: 100%|██████████| 53471/53471 [00:09<00:00, 5358.13 examples/s]


['left ventricular dysfunction', 'adhd', 'birth control']

---

## 📏 Step 8: Add a `review_length` Column with `map()`

Next, we want to know how long each review is (in words). We'll use `map()` to **add a new column** — not just update an existing one.

```python
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}
```

**How this creates a new column:**
- The function returns `{"review_length": ...}` — a key that **doesn't exist yet** in the dataset
- When `map()` sees a new key, it **automatically creates a new column** for it
- The original columns are preserved unchanged

**`"review".split()` explained:**
- `.split()` with no arguments splits on any whitespace (spaces, tabs, newlines)
- `len(...)` counts the resulting list of words
- This is a **rough heuristic** — it's not perfect (punctuation attached to words counts as part of the word), but it's fast and good enough

**After `map()`**, each row will have a new `review_length` field:
```python
{'patient_id': 206461, 'drugName': 'Valsartan', ..., 'review_length': 17}
```


In [ ]:
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}

In [ ]:
drug_dataset = drug_dataset.map(compute_review_length)
# Inspect the first training example
drug_dataset["train"][0]

Map: 100%|██████████| 53471/53471 [00:06<00:00, 8453.45 examples/s]


{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'left ventricular dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

---

## 🔃 Step 9: Sort by `review_length` to Inspect Extremes

Now that we have review lengths, let's use `Dataset.sort()` to find the shortest reviews.

```python
drug_dataset["train"].sort("review_length")[:3]
```

**What `sort()` does:**
- Returns a new `Dataset` sorted by the specified column (ascending by default)
- Does **not** modify the original dataset in place
- Slicing `[:3]` then grabs the first 3 rows (the shortest reviews)

**What we find:**  
Some reviews are just `"Excellent."`, `"useless"`, or `"ok"` — a single word! These very short reviews are **not useful** for training a model to predict medical conditions, because they carry no meaningful information about the drug or the condition.

> **`Dataset.sort()` vs Python's `sorted()`:**  
> `Dataset.sort()` works directly on Arrow-backed datasets without loading everything into memory. It's much more memory-efficient than converting to a list and sorting.


In [ ]:
drug_dataset["train"].sort("review_length")[:3]

{'patient_id': [111469, 13653, 53602],
 'drugName': ['Ledipasvir / sofosbuvir',
  'Amphetamine / dextroamphetamine',
  'Alesse'],
 'condition': ['hepatitis c', 'adhd', 'birth control'],
 'review': ['"Headache"', '"Great"', '"Awesome"'],
 'rating': [10.0, 10.0, 10.0],
 'date': ['February 3, 2015', 'October 20, 2009', 'November 23, 2015'],
 'usefulCount': [41, 3, 0],
 'review_length': [1, 1, 1]}

---

## ✂️ Step 10: Filter Out Short Reviews (< 30 words)

Very short reviews don't contain enough context. We'll remove any review with fewer than 30 words.

```python
drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
print(drug_dataset.num_rows)
```

**Why 30 words?**
- This is a practical threshold — reviews with fewer than 30 words are typically too vague to be useful for training
- The tutorial says this removes ~15% of reviews — a reasonable tradeoff between data quality and quantity

**`drug_dataset.num_rows`:**
- Returns a dictionary `{"train": N, "test": M}` showing row counts after filtering
- Quick sanity check to confirm the filter worked as expected

**Result:**
```
{'train': 136538, 'test': 45501}
```
Original was 160,398 train / 53,471 test → we dropped ~15% of short reviews.

> **Tip:** You can always try different thresholds (20, 40, 50 words) and evaluate how it affects your model's downstream performance.


In [ ]:
drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 32)
print(drug_dataset.num_rows)

Filter: 100%|██████████| 53471/53471 [00:00<00:00, 70189.72 examples/s]

{'train': 136538, 'test': 45501}


---

## 🌐 Step 11: Clean HTML Character Codes

Patient reviews on Drugs.com were scraped from the web, so they contain **HTML character entities** — special codes representing characters. For example:
- `&#039;` → `'` (apostrophe)
- `&amp;` → `&` (ampersand)
- `&lt;` → `<` (less than)

If left as-is, the tokenizer would treat `&#039;` as a sequence of tokens instead of a single apostrophe, corrupting the model's understanding of the text.

### Python's `html.unescape()`:
```python
import html
html.unescape("I&#039;m a transformer called BERT")
# → "I'm a transformer called BERT"
```

This function converts **all** HTML entities back to their original characters in one call. Let's first test it on a single string:


In [ ]:
import html

text = "I&#039;m a transformer called BERT"
html.unescape(text)

"I'm a transformer called BERT"

---

## 🗺️ Step 12: Apply HTML Unescaping via `map()` (Non-Batched)

Now we apply the unescaping to every review using `map()`:

```python
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})
```

**This is non-batched (default):** The lambda receives **one row at a time** as a dictionary, and returns the updated `review` field.

- `x` = one row dict
- `x["review"]` = the raw review string for that row
- `html.unescape(...)` = cleaned review string
- Returns `{"review": cleaned_string}` → `map()` updates just the review column

**Performance note:** This works correctly but is relatively slow because it processes one row at a time. We'll see a faster batched approach next.


In [ ]:
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

Map: 100%|██████████| 45501/45501 [00:21<00:00, 2125.98 examples/s]


---

## ⚡ Step 13: Batched `map()` — Process Multiple Rows at Once

The `batched=True` argument dramatically speeds up `map()` by sending **batches of rows** (default: 1,000) to your function instead of one row at a time.

```python
new_drug_dataset = drug_dataset.map(
    lambda x: {"review": [html.unescape(o) for o in x["review"]]},
    batched=True
)
```

**How batched mode changes the input:**

| Mode | `x["review"]` type | Your return type |
|------|--------------------|-----------------|
| Non-batched | `str` (single value) | `{"review": str}` |
| Batched | `list[str]` (1000 values) | `{"review": list[str]}` |

**Why it's faster:**
1. **List comprehensions** are faster than Python `for` loops — fewer overhead per iteration
2. **Reduced I/O overhead** — fetching 1,000 rows at once vs. 1,000 individual fetches
3. Some operations (like fast tokenizers) can **parallelize** across the batch internally

**Why does `batched=True` matter so much for tokenizers?**  
Fast tokenizers (backed by Rust) can process an entire list of texts **in parallel**, but they need the list all at once. With `batched=True`, you send them a list — without it, you send one string at a time and lose all parallelism.


In [ ]:
new_drug_dataset = drug_dataset.map(
    lambda x: {"review": [html.unescape(o) for o in x["review"]]}, batched=True
)

Map: 100%|██████████| 45501/45501 [00:00<00:00, 136297.53 examples/s]


---

## 🔤 Step 14: Tokenization with a Fast Tokenizer

Tokenization converts raw text into **token IDs** that a model can understand. Here we use BERT's tokenizer.

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    return tokenizer(examples["review"], truncation=True)
```

**What `AutoTokenizer` does:**
- Automatically detects the correct tokenizer class for the model (`bert-base-cased` → `BertTokenizerFast`)
- `from_pretrained()` downloads the tokenizer's vocabulary and config files
- `use_fast=True` (default) → uses the **Rust-backed fast tokenizer**

**`truncation=True` explained:**
- BERT has a maximum sequence length of 512 tokens
- `truncation=True` automatically cuts off any review longer than 512 tokens
- Without this, very long reviews would cause an error

**`bert-base-cased` vs `bert-base-uncased`:**
- **cased**: Distinguishes uppercase/lowercase — `"Apple"` ≠ `"apple"` — better for NER, proper nouns
- **uncased**: Lowercases everything first — smaller vocabulary, slightly simpler

**What the tokenizer returns:**
```python
{
    "input_ids": [[101, 1000, ...], ...],      # Token IDs
    "token_type_ids": [[0, 0, ...], ...],       # Sentence A vs B (for BERT)
    "attention_mask": [[1, 1, ...], ...]        # 1=real token, 0=padding
}
```

> **`%time` magic:** Placing `%time` before a line in Jupyter measures and prints how long that single line takes to run. Very useful for benchmarking!


In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["review"], truncation=True)

In [ ]:
%time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 45501/45501 [00:15<00:00, 2917.53 examples/s]

CPU times: total: 3min 46s
Wall time: 54.2 s


---

## 🐢 Step 15: Multiprocessing with a Slow Tokenizer (`num_proc`)

`AutoTokenizer` by default returns a **fast tokenizer** (Rust-backed). For comparison, we can request a **slow tokenizer** (pure Python):

```python
def slow_tokenize_function(examples):
    global slow_tokenizer
    if 'slow_tokenizer' not in globals():
        from transformers import AutoTokenizer
        slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
    return slow_tokenizer(examples["review"], truncation=True)

tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=True, num_proc=8)
```

### Why `num_proc`?

When using a slow tokenizer, you lose the Rust parallelism. To compensate, `Dataset.map()` supports **Python multiprocessing** via `num_proc`:

- `num_proc=8` → spawns **8 worker processes**, each processing a shard of the dataset
- Each worker runs independently and processes its portion in parallel
- Results are merged back together at the end

**Performance comparison (approximate):**

| Method | Time |
|--------|------|
| Fast tokenizer, `batched=True` | ~54s |
| Slow tokenizer, `batched=True`, no `num_proc` | ~5 min |
| Slow tokenizer, `batched=True`, `num_proc=8` | ~1-2 min |

### ⚠️ Windows Multiprocessing Caveat

On **Windows**, `multiprocessing` uses the `spawn` method — each worker process is a **fresh Python interpreter** with no inherited variables or imports. This means:
- Variables defined in the Jupyter notebook (like a tokenizer) are **not available** in worker processes
- You must **import and initialize inside the function** so each worker can set up its own state

That's why we initialize `slow_tokenizer` inside `slow_tokenize_function` using `globals()` — each worker process initializes its own tokenizer on first call, then reuses it for subsequent batches.

> **Tip:** For fast tokenizers, `batched=True` alone is usually sufficient. Only add `num_proc` when using slow tokenizers or custom CPU-intensive processing functions.


In [10]:
def slow_tokenize_function(examples):
    global slow_tokenizer
    if 'slow_tokenizer' not in globals():
        from transformers import AutoTokenizer
        slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
    return slow_tokenizer(examples["review"], truncation=True)


tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=True, num_proc=8)

Map (num_proc=8):   0%|          | 0/161297 [00:00<?, ? examples/s]

Map (num_proc=8): 100%|██████████| 53766/53766 [00:40<00:00, 1315.21 examples/s]
